# 🚀 Codebase Analyst - Optimized Edition

**Key Improvements:**
- ✅ **Zero Code Duplication**: All classes defined once
- ✅ **Optimized Performance**: Cached models, singleton instances
- ✅ **Clean Structure**: 15 cells vs 49 original
- ✅ **100% Feature Coverage**: All advanced features included

**Features:**
- RAG-based Q&A with hybrid retrieval
- Knowledge graph & dependency analysis  
- Architecture pattern detection
- Security vulnerability scanning
- Prometheus monitoring & caching
- RAG evaluation with RAGAS
- Export to JSON/MD/HTML

**Quick Start:**
1. Run all cells in order
2. Edit config in Cell 3
3. Use Gradio UI at the end


In [ ]:
# @title 📦 Setup & Install

import sys, os, subprocess, getpass

# API Configuration
OPENAI_API_KEY_VAR = ""  # Paste your key here
AZURE_OPENAI_ENDPOINT = "https://aifoundry2212.cognitiveservices.azure.com/"
AZURE_API_VERSION = "2025-01-01-preview"
AZURE_MODEL = "gpt-test"
AZURE_DEPLOYMENT = "gpt-test"

print("Checking GPU...")
!nvidia-smi

# Set environment
if OPENAI_API_KEY_VAR.strip():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY_VAR.strip()
os.environ.update({
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_API_VERSION,
    "AZURE_OPENAI_MODEL": AZURE_MODEL,
    "AZURE_OPENAI_DEPLOYMENT": AZURE_DEPLOYMENT
})

!pip install -q python-dotenv

try:
    from google.colab import userdata
    from dotenv import load_dotenv
    load_dotenv('/content/.env', override=False)
except:
    from dotenv import load_dotenv
    load_dotenv()

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')

print("\nInstalling packages...")
def install(*p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *p])

packages = [
    "openai>=1.50.0", "sentence-transformers", "gradio", "tiktoken",
    "ragas", "datasets", "pandas", "numpy", "tqdm", "pydantic>=2.10.0",
    "gitpython", "prometheus-client", "psutil", "networkx"
]

install("--upgrade", "pip")
install(*packages)

try:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tree-sitter-python"], capture_output=True)
except: pass
install("tree-sitter==0.21.*", "tree-sitter-languages==1.10.2")

print("✅ Setup complete!")


In [ ]:
# @title ⚙️ Configuration & Imports

import os, time, pickle, shutil, hashlib, re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional
from collections import Counter, OrderedDict
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
from openai import AzureOpenAI

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except: 
    SENTENCE_TRANSFORMERS_AVAILABLE = False

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    from datasets import Dataset
    RAGAS_AVAILABLE = True
except:
    RAGAS_AVAILABLE = False

from prometheus_client import Counter, Histogram, Gauge
import psutil

@dataclass
class Config:
    # Repository (EDIT THIS)
    repo_url: str = "https://github.com/gkamradt/langchain-tutorials.git"
    repo_name: str = "langchain-tutorials"
    
    # Paths
    data_dir: Path = Path("/content")
    cache_dir: Path = Path("/content/cache")
    index_dir: Path = Path("/content/index")
    
    # Settings
    chunk_max_lines: int = 50
    batch_size: int = 32
    embedding_model: str = "all-MiniLM-L6-v2"
    dense_weight: float = 0.6
    sparse_weight: float = 0.4
    
    # Azure OpenAI
    azure_api_key: str = os.getenv("OPENAI_API_KEY", "")
    azure_endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT", "")
    azure_api_version: str = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
    azure_model: str = os.getenv("AZURE_OPENAI_MODEL", "gpt-test")
    azure_deployment: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-test")
    
    def ensure_dirs(self):
        for d in [self.cache_dir, self.index_dir]:
            d.mkdir(parents=True, exist_ok=True)

config = Config()
config.ensure_dirs()
print(f"✅ Config: {config.repo_name}")


In [ ]:
# @title 📊 Monitoring & Advanced Caching

"""
Prometheus-based monitoring for the Codebase Analyst.
Tracks requests, latency, cache hits, and system metrics.
"""
from prometheus_client import Counter, Histogram, Gauge, REGISTRY
import psutil
import time
from typing import Optional

# Request Metrics
query_counter = Counter(
    'codebase_analyst_queries_total',
    'Total number of queries processed',
    ['status']
)

query_latency = Histogram(
    'codebase_analyst_query_latency_seconds',
    'Query processing latency in seconds',
    buckets=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 30.0]
)

llm_latency = Histogram(
    'codebase_analyst_llm_latency_seconds',
    'LLM call latency in seconds',
    buckets=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
)

# Cache Metrics
cache_hits = Counter('codebase_analyst_cache_hits_total', 'Number of cache hits')
cache_misses = Counter('codebase_analyst_cache_misses_total', 'Number of cache misses')

# Retrieval Metrics
retrieval_latency = Histogram(
    'codebase_analyst_retrieval_latency_seconds',
    'Retrieval latency in seconds'
)

chunks_retrieved = Histogram(
    'codebase_analyst_chunks_retrieved',
    'Number of chunks retrieved per query',
    buckets=[1, 3, 5, 10, 20, 50]
)

# System Metrics
cpu_usage = Gauge('codebase_analyst_cpu_percent', 'CPU usage percentage')
memory_usage = Gauge('codebase_analyst_memory_mb', 'Memory usage in MB')

class MetricsCollector:
    """Centralized metrics collection"""
    
    def __init__(self):
        self.start_time = time.time()
        
    def record_query(self, status: str = 'success'):
        """Record a query with status"""
        query_counter.labels(status=status).inc()
    
    def record_query_latency(self, latency: float):
        """Record query processing time"""
        query_latency.observe(latency)
    
    def record_llm_latency(self, latency: float):
        """Record LLM call time"""
        llm_latency.observe(latency)
    
    def record_cache_hit(self):
        """Record cache hit"""
        cache_hits.inc()
    
    def record_cache_miss(self):
        """Record cache miss"""
        cache_misses.inc()
    
    def record_retrieval(self, latency: float, num_chunks: int):
        """Record retrieval metrics"""
        retrieval_latency.observe(latency)
        chunks_retrieved.observe(num_chunks)
    
    def update_system_metrics(self):
        """Update system resource metrics"""
        cpu_usage.set(psutil.cpu_percent())
        memory_usage.set(psutil.Process().memory_info().rss / 1024 / 1024)
    
    def get_cache_hit_rate(self) -> float:
        """Calculate cache hit rate"""
        hits = cache_hits._value.get()
        misses = cache_misses._value.get()
        total = hits + misses
        return hits / total if total > 0 else 0.0

# Global metrics instance
metrics = MetricsCollector()


"""
Advanced cache manager with prefix-based caching and LRU eviction.
Complements the semantic cache with structured key management.
"""
import hashlib
import time
from typing import Any, Optional, Dict
from collections import OrderedDict

class CachedPrefix:
    """Represents a cached prefix for faster lookups"""
    def __init__(self, prefix: str, value: Any, ttl: Optional[int] = None):
        self.prefix = prefix
        self.value = value
        self.created_at = time.time()
        self.ttl = ttl
        self.access_count = 0
        self.last_accessed = time.time()
    
    def is_expired(self) -> bool:
        if self.ttl is None:
            return False
        return (time.time() - self.created_at) > self.ttl
    
    def access(self):
        self.access_count += 1
        self.last_accessed = time.time()

class CacheManager:
    """
    Advanced cache manager with:
    - Prefix-based caching for common query patterns
    - LRU eviction policy
    - TTL support
    - Statistics tracking
    """
    
    def __init__(self, max_size: int = 1000, default_ttl: Optional[int] = 3600):
        self.max_size = max_size
        self.default_ttl = default_ttl
        self.cache: OrderedDict[str, CachedPrefix] = OrderedDict()
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, key: str) -> str:
        """Generate cache key"""
        return hashlib.md5(key.encode()).hexdigest()
    
    def get(self, key: str) -> Optional[Any]:
        """Get value from cache"""
        cache_key = self._make_key(key)
        
        if cache_key in self.cache:
            entry = self.cache[cache_key]
            
            if entry.is_expired():
                del self.cache[cache_key]
                self.misses += 1
                return None
            
            # Move to end (LRU)
            self.cache.move_to_end(cache_key)
            entry.access()
            self.hits += 1
            return entry.value
        
        self.misses += 1
        return None
    
    def set(self, key: str, value: Any, ttl: Optional[int] = None):
        """Set value in cache"""
        cache_key = self._make_key(key)
        
        # Evict oldest if at capacity
        if len(self.cache) >= self.max_size and cache_key not in self.cache:
            self.cache.popitem(last=False)
        
        ttl = ttl if ttl is not None else self.default_ttl
        self.cache[cache_key] = CachedPrefix(key, value, ttl)
        self.cache.move_to_end(cache_key)
    
    def invalidate(self, key: str):
        """Invalidate specific cache entry"""
        cache_key = self._make_key(key)
        if cache_key in self.cache:
            del self.cache[cache_key]
    
    def invalidate_prefix(self, prefix: str):
        """Invalidate all entries matching prefix"""
        to_remove = [k for k, v in self.cache.items() if v.prefix.startswith(prefix)]
        for k in to_remove:
            del self.cache[k]
    
    def clear(self):
        """Clear entire cache"""
        self.cache.clear()
        self.hits = 0
        self.misses = 0
    
    def get_stats(self) -> Dict[str, Any]:
        """Get cache statistics"""
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0.0
        
        return {
            'size': len(self.cache),
            'max_size': self.max_size,
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': hit_rate,
            'total_requests': total
        }


print("✅ Monitoring & caching ready")


In [ ]:
# @title 🔬 Analysis Tools (Knowledge Graph, Architecture, Security)

"""
Code knowledge graph and impact analysis using NetworkX.
Tracks dependencies, function calls, and analyzes change impact.
"""
import networkx as nx
from typing import List, Dict, Any, Set
from pathlib import Path
import re

class CodeKnowledgeGraph:
    """Build and query dependency graphs from parsed code"""
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.file_map: Dict[str, Dict[str, Any]] = {}
    
    def add_file(self, file_path: str, functions: List[Dict], classes: List[Dict], imports: List[str]):
        """Add file node with its components"""
        self.file_map[file_path] = {
            'functions': functions,
            'classes': classes,
            'imports': imports
        }
        
        # Add file node
        self.graph.add_node(file_path, type='file')
        
        # Add function nodes
        for func in functions:
            func_id = f"{file_path}::{func['name']}"
            self.graph.add_node(func_id, type='function', **func)
            self.graph.add_edge(file_path, func_id, relationship='contains')
        
        # Add class nodes
        for cls in classes:
            cls_id = f"{file_path}::{cls['name']}"
            self.graph.add_node(cls_id, type='class', **cls)
            self.graph.add_edge(file_path, cls_id, relationship='contains')
            
            # Add methods
            for method in cls.get('methods', []):
                method_id = f"{cls_id}.{method}"
                self.graph.add_node(method_id, type='method', name=method)
                self.graph.add_edge(cls_id, method_id, relationship='has_method')
        
        # Add import edges
        for imp in imports:
            # Try to find imported module in our graph
            potential_files = [f for f in self.file_map.keys() if imp in f]
            for imp_file in potential_files:
                self.graph.add_edge(file_path, imp_file, relationship='imports')
    
    def build_from_chunks(self, chunks: List[Dict[str, Any]]):
        """Build graph from parsed chunks"""
        # Group by file
        files_data = {}
        for chunk in chunks:
            fp = chunk['file_path']
            if fp not in files_data:
                files_data[fp] = {'functions': [], 'classes': []}
            
            files_data[fp]['functions'].extend(chunk.get('functions', []))
            files_data[fp]['classes'].extend(chunk.get('classes', []))
        
        # Add each file (simplified - no import tracking here)
        for fp, data in files_data.items():
            self.add_file(fp, data['functions'], data['classes'], [])
    
    def get_dependencies(self, file_path: str) -> List[str]:
        """Get files that this file depends on"""
        if file_path not in self.graph:
            return []
        return [n for n in self.graph.successors(file_path) if self.graph.nodes[n].get('type') == 'file']
    
    def get_dependents(self, file_path: str) -> List[str]:
        """Get files that depend on this file"""
        if file_path not in self.graph:
            return []
        return [n for n in self.graph.predecessors(file_path) if self.graph.nodes[n].get('type') == 'file']
    
    def find_central_files(self, top_k: int = 10) -> List[tuple]:
        """Find most central/important files using PageRank"""
        file_subgraph = self.graph.subgraph([n for n in self.graph.nodes if self.graph.nodes[n].get('type') == 'file'])
        pagerank = nx.pagerank(file_subgraph)
        return sorted(pagerank.items(), key=lambda x: -x[1])[:top_k]

class ImpactAnalyzer:
    """Analyze impact of code changes using the knowledge graph"""
    
    def __init__(self, knowledge_graph: CodeKnowledgeGraph):
        self.kg = knowledge_graph
    
    def analyze_file_impact(self, file_path: str) -> Dict[str, Any]:
        """Analyze impact if a file is modified"""
        direct_dependents = self.kg.get_dependents(file_path)
        
        # Get transitive closure (all affected files)
        affected = set()
        to_visit = set(direct_dependents)
        visited = set()
        
        while to_visit:
            current = to_visit.pop()
            if current in visited:
                continue
            visited.add(current)
            affected.add(current)
            
            deps = self.kg.get_dependents(current)
            to_visit.update(deps)
        
        return {
            'file': file_path,
            'direct_dependents': len(direct_dependents),
            'total_affected_files': len(affected),
            'affected_files': list(affected)[:20],  # Limit for display
            'risk_level': self._calculate_risk(len(affected))
        }
    
    def _calculate_risk(self, num_affected: int) -> str:
        """Calculate risk level based on number of affected files"""
        if num_affected == 0:
            return 'low'
        elif num_affected < 5:
            return 'medium'
        else:
            return 'high'
    
    def find_critical_paths(self, from_file: str, to_file: str) -> List[List[str]]:
        """Find dependency paths between two files"""
        try:
            paths = list(nx.all_simple_paths(self.kg.graph, from_file, to_file, cutoff=5))
            return paths[:5]  # Limit to 5 paths
        except (nx.NodeNotFound, nx.NetworkXNoPath):
            return []


"""
Architecture pattern detection and analysis.
Identifies common patterns like MVC, layered architecture, microservices, etc.
"""
from typing import List, Dict, Any
from collections import Counter
import re

class ArchitectureAnalyzer:
    """Detect and analyze architectural patterns in codebases"""
    
    def __init__(self, chunks: List[Dict[str, Any]]):
        self.chunks = chunks
        self.files = list(set(c['file_path'] for c in chunks))
    
    def detect_patterns(self) -> Dict[str, Any]:
        """Detect common architectural patterns"""
        patterns = {
            'mvc': self._detect_mvc(),
            'layered': self._detect_layered(),
            'microservices': self._detect_microservices(),
            'monolithic': self._detect_monolithic(),
        }
        
        # Determine primary pattern
        pattern_scores = {k: v['confidence'] for k, v in patterns.items() if v['detected']}
        primary = max(pattern_scores.items(), key=lambda x: x[1])[0] if pattern_scores else 'unknown'
        
        return {
            'primary_pattern': primary,
            'detected_patterns': patterns,
            'file_count': len(self.files),
            'component_analysis': self._analyze_components()
        }
    
    def _detect_mvc(self) -> Dict[str, Any]:
        """Detect MVC pattern"""
        mvc_indicators = {
            'models': 0,
            'views': 0,
            'controllers': 0
        }
        
        for f in self.files:
            f_lower = f.lower()
            if 'model' in f_lower:
                mvc_indicators['models'] += 1
            if 'view' in f_lower or 'template' in f_lower:
                mvc_indicators['views'] += 1
            if 'controller' in f_lower or 'handler' in f_lower:
                mvc_indicators['controllers'] += 1
        
        detected = all(count > 0 for count in mvc_indicators.values())
        confidence = sum(mvc_indicators.values()) / len(self.files) if self.files else 0
        
        return {
            'detected': detected,
            'confidence': min(confidence, 1.0),
            'details': mvc_indicators
        }
    
    def _detect_layered(self) -> Dict[str, Any]:
        """Detect layered architecture"""
        layers = {
            'presentation': 0,
            'business': 0,
            'data': 0,
            'infrastructure': 0
        }
        
        for f in self.files:
            f_lower = f.lower()
            if any(x in f_lower for x in ['ui', 'views', 'frontend', 'presentation']):
                layers['presentation'] += 1
            if any(x in f_lower for x in ['service', 'business', 'logic', 'domain']):
                layers['business'] += 1
            if any(x in f_lower for x in ['data', 'repository', 'dao', 'database']):
                layers['data'] += 1
            if any(x in f_lower for x in ['util', 'helper', 'config', 'infrastructure']):
                layers['infrastructure'] += 1
        
        detected = sum(1 for c in layers.values() if c > 0) >= 3
        confidence = sum(1 for c in layers.values() if c > 0) / 4
        
        return {
            'detected': detected,
            'confidence': confidence,
            'details': layers
        }
    
    def _detect_microservices(self) -> Dict[str, Any]:
        """Detect microservices architecture"""
        service_indicators = []
        
        for f in self.files:
            if 'service' in f.lower() and ('api' in f.lower() or 'grpc' in f.lower()):
                service_indicators.append(f)
        
        detected = len(service_indicators) >= 2
        confidence = min(len(service_indicators) / 5, 1.0)
        
        return {
            'detected': detected,
            'confidence': confidence,
            'details': {'service_count': len(service_indicators)}
        }
    
    def _detect_monolithic(self) -> Dict[str, Any]:
        """Detect monolithic architecture"""
        # Simplified: large number of files in a single directory structure
        detected = len(self.files) > 50
        confidence = min(len(self.files) / 200, 1.0)
        
        return {
            'detected': detected,
            'confidence': confidence,
            'details': {'file_count': len(self.files)}
        }
    
    def _analyze_components(self) -> Dict[str, Any]:
        """Analyze codebase components"""
        extensions = Counter()
        for f in self.files:
            ext = f.split('.')[-1] if '.' in f else 'no_extension'
            extensions[ext] += 1
        
        return {
            'file_types': dict(extensions.most_common(10)),
            'total_files': len(self.files)
        }
    
    def get_summary(self) -> str:
        """Get human-readable architecture summary"""
        analysis = self.detect_patterns()
        pattern = analysis['primary_pattern']
        
        summaries = {
            'mvc': "MVC (Model-View-Controller) pattern detected",
            'layered': "Layered architecture with separation of concerns",
            'microservices': "Microservices architecture with distributed services",
            'monolithic': "Monolithic architecture with centralized structure",
            'unknown': "Architecture pattern unclear from file structure"
        }
        
        return summaries.get(pattern, "Unknown architecture")


"""
Security vulnerability scanner.
Detects common security issues and anti-patterns.
"""
from typing import List, Dict, Any
import re

class SecurityAnalyzer:
    """Scan code for potential security vulnerabilities"""
    
    def __init__(self, chunks: List[Dict[str, Any]]):
        self.chunks = chunks
        self.vulnerability_patterns = {
            'hardcoded_secrets': {
                'patterns': [
                    r'password\s*=\s*["\']([^"\']+)["\']',
                    r'api_key\s*=\s*["\']([^"\']+)["\']',
                    r'secret\s*=\s*["\']([^"\']+)["\']',
                    r'token\s*=\s*["\']([^"\']+)["\']',
                ],
                'severity': 'high',
                'description': 'Hardcoded credentials detected'
            },
            'sql_injection': {
                'patterns': [
                    r'execute\([^)]*%s',
                    r'cursor\.execute\([^)]*\+',
                    r'\.format\s*\(',
                ],
                'severity': 'high',
                'description': 'Potential SQL injection vulnerability'
            },
            'insecure_random': {
                'patterns': [
                    r'import\s+random\b',
                    r'random\.random',
                    r'Math\.random',
                ],
                'severity': 'medium',
                'description': 'Use of insecure random number generator'
            },
            'eval_usage': {
                'patterns': [
                    r'\beval\s*\(',
                    r'\bexec\s*\(',
                ],
                'severity': 'high',
                'description': 'Dangerous use of eval/exec'
            },
            'debug_mode': {
                'patterns': [
                    r'DEBUG\s*=\s*True',
                    r'debug\s*=\s*True',
                ],
                'severity': 'medium',
                'description': 'Debug mode enabled'
            },
        }
    
    def scan(self) -> Dict[str, Any]:
        """Perform security scan"""
        findings = []
        
        for chunk in self.chunks:
            content = chunk['content']
            file_path = chunk['file_path']
            
            for vuln_type, config in self.vulnerability_patterns.items():
                for pattern in config['patterns']:
                    matches = re.finditer(pattern, content, re.IGNORECASE)
                    for match in matches:
                        findings.append({
                            'type': vuln_type,
                            'severity': config['severity'],
                            'description': config['description'],
                            'file': file_path,
                            'line': content[:match.start()].count('\n') + chunk.get('start_line', 1),
                            'snippet': match.group(0)[:100]
                        })
        
        return {
            'total_findings': len(findings),
            'by_severity': self._group_by_severity(findings),
            'findings': findings[:50],  # Limit for display
            'risk_score': self._calculate_risk_score(findings)
        }
    
    def _group_by_severity(self, findings: List[Dict]) -> Dict[str, int]:
        """Group findings by severity"""
        severity_count = {'high': 0, 'medium': 0, 'low': 0}
        for f in findings:
            severity_count[f['severity']] += 1
        return severity_count
    
    def _calculate_risk_score(self, findings: List[Dict]) -> int:
        """Calculate overall risk score (0-100)"""
        weights = {'high': 10, 'medium': 5, 'low': 1}
        score = sum(weights.get(f['severity'], 1) for f in findings)
        return min(score, 100)
    
    def get_recommendations(self, findings: List[Dict]) -> List[str]:
        """Get security recommendations based on findings"""
        recommendations = []
        
        finding_types = set(f['type'] for f in findings)
        
        if 'hardcoded_secrets' in finding_types:
            recommendations.append("Move secrets to environment variables or secret management service")
        if 'sql_injection' in finding_types:
            recommendations.append("Use parameterized queries or ORM for database access")
        if 'insecure_random' in finding_types:
            recommendations.append("Use secrets.SystemRandom() for cryptographic operations")
        if 'eval_usage' in finding_types:
            recommendations.append("Avoid eval/exec; use safer alternatives like ast.literal_eval")
        if 'debug_mode' in finding_types:
            recommendations.append("Disable debug mode in production environments")
        
        return recommendations


print("✅ Analysis tools ready")


In [ ]:
# @title 📈 Evaluation & Export

"""
RAG Evaluation using RAGAS framework.
Measures faithfulness, relevance, and context recall.
"""
from typing import List, Dict, Any, Optional
import pandas as pd

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
    from datasets import Dataset
    RAGAS_AVAILABLE = True
except ImportError:
    RAGAS_AVAILABLE = False
    print("Warning: RAGAS not installed. Evaluation features limited.")

class RAGEvaluator:
    """Evaluate RAG quality using RAGAS metrics"""
    
    def __init__(self):
        if not RAGAS_AVAILABLE:
            raise ImportError("RAGAS library required. Install with: pip install ragas datasets")
        
        self.metrics = [
            faithfulness,
            answer_relevancy,
            context_recall,
            context_precision
        ]
    
    def evaluate_single(self, question: str, answer: str, contexts: List[str], 
                       ground_truth: Optional[str] = None) -> Dict[str, float]:
        """Evaluate a single QA pair"""
        
        data = {
            'question': [question],
            'answer': [answer],
            'contexts': [contexts],
        }
        
        if ground_truth:
            data['ground_truth'] = [ground_truth]
        
        dataset = Dataset.from_dict(data)
        
        # Use subset of metrics if no ground truth
        metrics_to_use = self.metrics if ground_truth else [faithfulness, answer_relevancy]
        
        results = evaluate(dataset, metrics=metrics_to_use)
        
        return {
            'faithfulness': results['faithfulness'],
            'answer_relevancy': results['answer_relevancy'],
            'context_recall': results.get('context_recall', None),
            'context_precision': results.get('context_precision', None)
        }
    
    def evaluate_batch(self, qa_pairs: List[Dict[str, Any]]) -> pd.DataFrame:
        """
        Evaluate multiple QA pairs
        
        Args:
            qa_pairs: List of dicts with 'question', 'answer', 'contexts', optional 'ground_truth'
        """
        questions = []
        answers = []
        contexts = []
        ground_truths = []
        has_ground_truth = all('ground_truth' in qa for qa in qa_pairs)
        
        for qa in qa_pairs:
            questions.append(qa['question'])
            answers.append(qa['answer'])
            contexts.append(qa['contexts'])
            if has_ground_truth:
                ground_truths.append(qa['ground_truth'])
        
        data = {
            'question': questions,
            'answer': answers,
            'contexts': contexts,
        }
        
        if has_ground_truth:
            data['ground_truth'] = ground_truths
        
        dataset = Dataset.from_dict(data)
        metrics_to_use = self.metrics if has_ground_truth else [faithfulness, answer_relevancy]
        
        results = evaluate(dataset, metrics=metrics_to_use)
        
        return pd.DataFrame(results)
    
    def get_summary_stats(self, results: pd.DataFrame) -> Dict[str, float]:
        """Get summary statistics from evaluation results"""
        return {
            'mean_faithfulness': results['faithfulness'].mean(),
            'mean_relevancy': results['answer_relevancy'].mean(),
            'mean_recall': results['context_recall'].mean() if 'context_recall' in results else None,
            'mean_precision': results['context_precision'].mean() if 'context_precision' in results else None,
        }


"""
Export analysis results to various formats (JSON, Markdown, HTML).
"""
import json
import shutil
from pathlib import Path
from typing import Dict, Any, List
from datetime import datetime

class ResultExporter:
    """Export query results and analysis to multiple formats"""
    
    def __init__(self, output_dir: Path = None):
        self.output_dir = output_dir or Path("exports")
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def export_to_json(self, data: Dict[str, Any], filename: str) -> Path:
        """Export data to JSON"""
        filepath = self.output_dir / f"{filename}.json"
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return filepath
    
    def export_to_markdown(self, query: str, answer: str, contexts: List[Dict], 
                          metadata: Dict = None, filename: str = "report") -> Path:
        """Export query result to Markdown"""
        filepath = self.output_dir / f"{filename}.md"
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"# Codebase Analysis Report\n\n")
            f.write(f"**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            if metadata:
                f.write(f"## Metadata\n\n")
                for key, value in metadata.items():
                    f.write(f"- **{key}**: {value}\n")
                f.write("\n")
            
            f.write(f"## Query\n\n```\n{query}\n```\n\n")
            f.write(f"## Answer\n\n{answer}\n\n")
            
            f.write(f"## Retrieved Context ({len(contexts)} chunks)\n\n")
            for i, ctx in enumerate(contexts, 1):
                payload = ctx.get('payload', ctx)
                f.write(f"### Context {i}\n\n")
                f.write(f"- **File**: `{payload['file_path']}`\n")
                f.write(f"- **Lines**: {payload['start_line']}-{payload['end_line']}\n")
                f.write(f"- **Score**: {ctx.get('score', 'N/A')}\n\n")
                f.write(f"```{payload.get('language', '')}\n")
                f.write(f"{payload['content']}\n")
                f.write(f"```\n\n")
        
        return filepath
    
    def export_to_html(self, query: str, answer: str, contexts: List[Dict],
                      metadata: Dict = None, filename: str = "report") -> Path:
        """Export query result to HTML"""
        filepath = self.output_dir / f"{filename}.html"
        
        html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Codebase Analysis Report</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; 
               max-width: 1200px; margin: 0 auto; padding: 20px; }}
        h1 {{ color: #2c3e50; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        .metadata {{ background: #f8f9fa; padding: 15px; border-radius: 5px; }}
        .query {{ background: #e8f4f8; padding: 15px; border-left: 4px solid #3498db; }}
        .answer {{ background: #e8f5e9; padding: 15px; border-left: 4px solid #4caf50; }}
        .context {{ background: #fff3cd; padding: 15px; margin: 10px 0; border-radius: 5px; }}
        pre {{ background: #f4f4f4; padding: 10px; overflow-x: auto; border-radius: 3px; }}
        code {{ font-family: 'Courier New', monospace; }}
        .file-info {{ color: #666; font-size: 0.9em; }}
    </style>
</head>
<body>
    <h1>🔍 Codebase Analysis Report</h1>
    <p><em>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</em></p>
"""
        
        if metadata:
            html += "<div class='metadata'><h2>Metadata</h2><ul>"
            for key, value in metadata.items():
                html += f"<li><strong>{key}</strong>: {value}</li>"
            html += "</ul></div>"
        
        html += f"""
    <h2>Query</h2>
    <div class='query'><code>{query}</code></div>
    
    <h2>Answer</h2>
    <div class='answer'>{answer.replace(chr(10), '<br>')}</div>
    
    <h2>Retrieved Context ({len(contexts)} chunks)</h2>
"""
        
        for i, ctx in enumerate(contexts, 1):
            payload = ctx.get('payload', ctx)
            html += f"""
    <div class='context'>
        <h3>Context {i}</h3>
        <div class='file-info'>
            <strong>File:</strong> {payload['file_path']}<br>
            <strong>Lines:</strong> {payload['start_line']}-{payload['end_line']}<br>
            <strong>Score:</strong> {ctx.get('score', 'N/A')}
        </div>
        <pre><code>{payload['content']}</code></pre>
    </div>
"""
        
        html += """
</body>
</html>
"""
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(html)
        
        return filepath
    
    def create_summary_report(self, results: List[Dict[str, Any]], filename: str = "summary") -> Dict[str, Path]:
        """Create summary report in multiple formats"""
        summary_data = {
            'total_queries': len(results),
            'generated_at': datetime.now().isoformat(),
            'results': results
        }
        
        json_path = self.export_to_json(summary_data, filename)
        
        return {
            'json': json_path,
            'output_dir': self.output_dir
        }


print("✅ Evaluation & export ready")


In [ ]:
# @title 🚀 Launch Application

# Initialize system
print("Initializing system...")
components = reindex_repository()

# Create analyst
hybrid_retriever = HybridRetriever(
    vector_store=components['vector_store'],
    sparse_retriever=components['sparse_retriever'],
    embedding_engine=components['embedding_engine']
)

code_tools = CodeTools(components['repo_dir'], components['all_chunks'])
cache = SemanticCache(components['embedding_engine'])

analyst = EnhancedCodebaseAnalyst(
    hybrid_retriever=hybrid_retriever,
    code_tools=code_tools,
    chunks=components['all_chunks'],
    cache_manager=cache
)

print("✅ System initialized!")

# Launch Gradio UI
demo = create_ui()
demo.launch(share=True)
